# 🎮 梦幻西游抓鬼任务 - YOLOv8 NPC 识别模型训练

## 数据集信息
- **图片数量**: 97 张标注图片
- **类别数量**: 7 个
- **类别列表**: 马副将、驿站老板、黑无常、钟馗、主鬼、小怪、鼠标指针

## 预计训练时间
- **GPU**: 5-10 分钟
- **CPU**: 30-60 分钟（不推荐）

### 1️⃣ 安装依赖

In [ ]:
!pip install ultralytics -q
print("✅ 依赖安装完成")

### 2️⃣ 上传数据集

In [ ]:
from google.colab import files
import zipfile
import os

print("📤 请上传 annotation_100.zip 文件...")
print("提示：在本地将 annotation_100 文件夹压缩为 ZIP 后上传")

uploaded = files.upload()

# 解压
for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"📂 正在解压 {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('dataset')
        print("✅ 解压完成")

# 创建数据配置文件
data_yaml = """
path: dataset/annotation_100
train: images
val: images

names:
  - 马副将
  - 驿站老板
  - 黑无常
  - 钟馗
  - 主鬼
  - 小怪
  - 鼠标指针
"""

with open('data.yaml', 'w') as f:
    f.write(data_yaml)

print("✅ 数据集配置完成")

### 3️⃣ 检查硬件设备

In [ ]:
import torch
import ultralytics

print(f"🔥 PyTorch 版本：{torch.__version__}")
print(f"🤖 Ultralytics 版本：{ultralytics.__version__}")
print(f"💻 设备：{'✅ GPU (CUDA)' if torch.cuda.is_available() else '⚠️ CPU'}")

if torch.cuda.is_available():
    print(f"🎮 GPU 型号：{torch.cuda.get_device_name(0)}")
    print(f"📊 GPU 内存：{torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

### 4️⃣ 开始训练 🚀

In [ ]:
from ultralytics import YOLO

# 加载预训练模型
print("📥 加载 YOLOv8n 预训练模型...")
model = YOLO('yolov8n.pt')

# 开始训练
print("🚀 开始训练...")
results = model.train(
    data='data.yaml',
    epochs=100,      # 训练 100 轮
    batch=16,        # 批次大小
    imgsz=640,       # 图片大小
    device=0 if torch.cuda.is_available() else 'cpu',
    workers=2,
    project='trained_model',
    name='zhuagui_v1',
    verbose=True,
    plots=True
)

print("✅ 训练完成！")

### 5️⃣ 验证模型性能

In [ ]:
print("📊 验证模型性能...")
metrics = model.val()

print("\n" + "="*50)
print("📈 训练结果")
print("="*50)
print(f"mAP50:      {metrics.box.map50:.4f}")
print(f"mAP50-95:   {metrics.box.map:.4f}")
print(f"精确率：    {metrics.box.mp:.4f}")
print(f"召回率：    {metrics.box.mr:.4f}")
print("="*50)

### 6️⃣ 导出模型

In [ ]:
print("📦 导出模型...")

# 导出为 ONNX 格式
onnx_path = model.export(format='onnx')
print(f"✅ ONNX 模型：{onnx_path}")

# 导出为 TorchScript 格式
torchscript_path = model.export(format='torchscript')
print(f"✅ TorchScript 模型：{torchscript_path}")

print("\n✅ 模型导出完成！")

### 7️⃣ 下载训练好的模型

In [ ]:
from google.colab import files
import os

print("📥 准备下载模型...")

# 模型路径
model_path = 'trained_model/zhuagui_v1/weights/best.pt'
onnx_path = 'trained_model/zhuagui_v1/weights/best.onnx'

if os.path.exists(model_path):
    print(f"📦 下载 PyTorch 模型：{model_path}")
    files.download(model_path)
    
    print(f"📦 下载 ONNX 模型：{onnx_path}")
    files.download(onnx_path)
    
    print("✅ 下载完成！")
else:
    print("❌ 模型文件未找到")

### 8️⃣ 测试模型（可选）

In [ ]:
# 上传测试图片
print("📤 上传测试图片...")
test_images = files.upload()

# 使用训练好的模型进行预测
for img_path in test_images.keys():
    print(f"\n🔍 预测：{img_path}")
    results = model(img_path)
    
    # 显示结果
    results[0].show()
    
    # 打印检测结果
    boxes = results[0].boxes
    if len(boxes) > 0:
        print(f"📊 检测到 {len(boxes)} 个目标:")
        for box in boxes:
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            cls_name = model.names[cls]
            print(f"  - {cls_name}: {conf:.2%}")
    else:
        print("  未检测到目标")

## 🎉 完成！

现在你可以：
1. 下载训练好的模型（best.pt 和 best.onnx）
2. 将模型集成到抓鬼自动化系统中
3. 测试 NPC 识别效果

如有问题，请查看训练日志或重新调整参数。